[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB10_project_template.ipynb)

# NB10 · Project notebook (template)
**Artificial Intelligence in Medical Imaging** · Days 4–5 · one notebook per team · due **Sunday 22 November 2026, 23:59 (Madrid), uploaded to Moodle**

This template **runs top to bottom on PneumoniaMNIST with a small CNN once the `TODO(team)` cells are filled in** (the solutions notebook is a complete working example). Your job is to turn it into *your* project: change the question, the data or the model, keep the structure.

| Section | Day | You deliver |
|---|---|---|
| 0 · Team and question | 4 | names, roles, one question that a result can answer |
| 1 · Data and exploration | 4 | class table, 8 images, sizes, missing labels |
| 2 · Split | 4 | leakage-safe split **with an assertion**, saved to `outputs/split.json` |
| 3 · Metric and trivial baseline | 4 | metric chosen *before* looking, trivial baseline |
| 4 · Model and first result | 4 | fixed seed, validation metric with interval |
| 5 · Test set, once | 4–5 | one evaluation, logged |
| 6 · Error analysis | 4 | the 10 most confident wrong predictions + 1 sentence per observation |
| 7 · Explanation and sanity check | 5 | Grad-CAM + weight randomisation |
| 8 · Shift or subgroup check | 5 | table with intervals |
| 9 · Model card and checklist | 5 | `outputs/model_card.md`, self-check |

**Graded criteria (25% each):** correctness · code clarity · proper use of AI techniques · thoughtful interpretation. Honest and modest beats impressive and wrong. **AI tools:** only in the ways the instructor states in class; you must be able to explain every line; declare any use in one line in section 0.

## 0 · Team and question

In [ ]:
TEAM = {
    "team_number": "TODO(team)",
    "members": {"TODO(team) name": "clinical lead", "TODO(team) name 2": "data lead"},     # one entry per member with the role of today
    "ai_tools_used": "TODO(team) none, or one line",
}
QUESTION = "TODO(team) one sentence that a result can answer"
print(TEAM["team_number"], "|", QUESTION)

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, json, hashlib, copy
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
SEED = 42; set_seed(SEED); Path("data").mkdir(exist_ok=True); Path("outputs").mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", device)

DATASET = "PneumoniaMNIST"      # any BINARY MedMNIST+ dataset, e.g. BreastMNIST, PneumoniaMNIST
SIZE = 28                       # 28, 64, 128 or 224 (MedMNIST+)
EPOCHS = 10                     # raise on GPU if you need to
LABEL_FRACTION = 1.0            # question A: 1.0, 0.1, 0.01

## 1 · Data and exploration
Where do the data come from, how many images, who is missing? Look at the **class table**, **eight images**, the **sizes** and any **missing labels** before you model anything.

In [ ]:
def load_medmnist(name, size):
    import medmnist
    from medmnist import INFO
    info = INFO[name.lower()]; cls = getattr(medmnist, info["python_class"]); kw = {} if size == 28 else {"size": size}
    out = {}
    for s in ("train", "val", "test"):
        d = cls(split=s, download=True, root="data", **kw)
        out[s] = (d.imgs.reshape(len(d.imgs), size, size).astype(np.uint8) if d.imgs.ndim <= 3 or d.imgs.shape[-1] == 1 else d.imgs.mean(-1).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
    return out, info

raw, INFO_ = load_medmnist(DATASET, SIZE)
labels_all = np.concatenate([v[1] for v in raw.values()])
assert set(np.unique(labels_all)) <= {0, 1}, f"{DATASET} is not binary ({INFO_['label']}); choose a binary dataset or adapt the code"
print(DATASET, "| classes:", INFO_["label"], "| licence:", INFO_.get("license", "see the MedMNIST page"))
X = {s: torch.from_numpy(v[0]).float().div(255).unsqueeze(1) for s, v in raw.items()}; Y = {s: torch.from_numpy(v[1]) for s, v in raw.items()}
table = pd.DataFrame({s: {"n": len(Y[s]), "positive": int(Y[s].sum()), "negative": int((Y[s] == 0).sum()), "positive_share": round(float(Y[s].float().mean()), 3)} for s in X}).T
print(table)
print("image size:", tuple(X["train"].shape[1:]), "| missing labels:", int(np.isnan(labels_all.astype(float)).sum()))
fig, ax = plt.subplots(2, 4, figsize=(8, 4.2))
for k, a in enumerate(ax.ravel()): a.imshow(X["train"][k, 0], cmap="gray"); a.set_title(f"label {int(Y['train'][k])}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

**TODO(team):** in a cell below, write 3 sentences: who is *missing* from this dataset? (Where were the images collected, which population, how were labels made?) What is the prevalence, and what does it do to the metric you will choose?

## 2 · Split: leakage-safe, with an assertion
- **MedMNIST+ ships its own official split:** use it and do **not** re-split. The paper does not state that it is by patient, so say that in your limitations.
- **If your data has a patient or study identifier**, split by it (`GroupShuffleSplit`) and keep the assertion.
- Check the class table after splitting. Augment **after** splitting, never before.
- **Real finding:** PneumoniaMNIST contains a few *identical* images in train and val. The cell below counts them and removes them from the evaluation sets (train is never touched). Run the cell once; after a rerun the "found" count shows 0 because they are already removed. Report the numbers in your model card.

In [ ]:
# The recipe for data WITH a patient identifier (demonstrated on synthetic groups, so you can copy it)
patient_id_demo = np.repeat(np.arange(200), 5); y_demo = np.random.default_rng(0).integers(0, 2, len(patient_id_demo))
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_i, te_i = next(gss.split(np.zeros(len(y_demo)), y_demo, groups=patient_id_demo))
assert set(patient_id_demo[tr_i]).isdisjoint(patient_id_demo[te_i]), "a patient is in both sets"
print("grouped split demo: train", len(tr_i), "| test", len(te_i), "| shared patients: 0")

def image_hashes(arr_uint8): return {hashlib.md5(a.tobytes()).hexdigest() for a in arr_uint8}
def count_dups(r):
    h = {k: image_hashes(r[k][0]) for k in r}
    return {"train&val": len(h["train"] & h["val"]), "train&test": len(h["train"] & h["test"]), "val&test": len(h["val"] & h["test"])}
dups_found = count_dups(raw)
print("exact duplicate images across sets (as downloaded):", dups_found)
# Real finding: PneumoniaMNIST contains a few identical images in train and val. Rule: the EVALUATION sets lose them (train is never touched).
h_tr = image_hashes(raw["train"][0])
keep_val = np.array([hashlib.md5(a.tobytes()).hexdigest() not in h_tr for a in raw["val"][0]])
h_va = image_hashes(raw["val"][0][keep_val])
keep_test = np.array([hashlib.md5(a.tobytes()).hexdigest() not in (h_tr | h_va) for a in raw["test"][0]])
removed = {"val": int((~keep_val).sum()), "test": int((~keep_test).sum())}
for k, keep in (("val", keep_val), ("test", keep_test)):
    raw[k] = (raw[k][0][keep], raw[k][1][keep]); X[k] = X[k][torch.from_numpy(keep)]; Y[k] = Y[k][torch.from_numpy(keep)]
dups = count_dups(raw)
print("removed from the evaluation sets:", removed, "| duplicates after cleaning:", dups)
assert sum(dups.values()) == 0, "identical images still appear in two sets"

split_info = {"mode": "official MedMNIST+ split", "seed": SEED, "dataset": DATASET, "size": SIZE, "n": {s: int(len(Y[s])) for s in Y}, "positive": {s: int(Y[s].sum()) for s in Y}, "exact_duplicates_found_before_cleaning": dups_found, "removed_from_evaluation_sets": removed}
json.dump(split_info, open("outputs/split.json", "w"), indent=1); print("saved outputs/split.json")

## 3 · Metric and trivial baseline
**Choose the metric before you look at results**, and justify it in one sentence (prevalence! clinical cost of a miss vs a false alarm). Then compute what a model that does nothing intelligent scores: **always the majority class**.

In [ ]:
MAIN_METRIC = "TODO(team) e.g. AUC with interval; sensitivity/specificity; Dice"
METRIC_WHY = "TODO(team) one sentence: why this metric for this question and this prevalence"
REQUIRED_SENSITIVITY = 0.90      # TODO(team) change or delete if your question has no such requirement

# TODO(team): the trivial baseline. majority class of the TRAIN set, and its accuracy on validation
maj_class = 0
maj_acc_val = 0.0
trivial = {"majority class": maj_class, "accuracy of always predicting it (val)": round(maj_acc_val, 3), "AUC of a constant score": 0.5}
print(trivial)
assert abs(maj_acc_val - max(Y["val"].float().mean().item(), 1 - Y["val"].float().mean().item())) < 1e-9, "compute the accuracy of always predicting the majority class"

## 4 · Model and first result (fixed seed, validation only)

In [ ]:
def build_cnn():
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 1))
LAST_CONV_RELU = 7              # index of the last ReLU in build_cnn(): the layer Grad-CAM will use

def stratified_subset(y, frac, seed=SEED):
    if frac >= 1.0: return np.arange(len(y))
    from sklearn.model_selection import train_test_split
    return np.sort(train_test_split(np.arange(len(y)), train_size=frac, stratify=y, random_state=seed)[0])

@torch.no_grad()
def predict(model, x, bs=512):
    model.eval(); return torch.cat([torch.sigmoid(model(x[i:i + bs].to(device))).cpu() for i in range(0, len(x), bs)]).numpy().ravel()

def train(model, epochs=EPOCHS, lr=1e-3, frac=LABEL_FRACTION, seed=SEED, Xtr=None):
    set_seed(seed); model.to(device); idx = stratified_subset(Y["train"].numpy(), frac, seed)
    Xt = (X["train"] if Xtr is None else Xtr)[idx]; Yt = Y["train"][idx]
    loader = DataLoader(TensorDataset(Xt, Yt), batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.Adam(model.parameters(), lr=lr); crit = nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.float().unsqueeze(1).to(device); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    return model

def bootstrap_ci(metric_fn, scores, labels, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, len(labels), len(labels))
        if labels[i].min() == labels[i].max(): continue
        vals.append(metric_fn(labels[i], scores[i]))
    return np.percentile(vals, [2.5, 97.5])

t0 = time.time(); model = train(build_cnn())
p_val = predict(model, X["val"]); y_val = Y["val"].numpy()
auc_val = roc_auc_score(y_val, p_val); lo, hi = bootstrap_ci(roc_auc_score, p_val, y_val)
print(f"validation AUC {auc_val:.3f}  95% interval [{lo:.3f}, {hi:.3f}] | trivial AUC 0.5 | {(time.time() - t0) / 60:.1f} min")
assert auc_val > 0.5, "the model should beat a constant score on validation"

**Threshold from the requirement (on validation).** Highest specificity among thresholds with sensitivity ≥ `REQUIRED_SENSITIVITY`.

In [ ]:
fpr, tpr, thr = roc_curve(y_val, p_val)
idx = int(np.argmax(tpr >= REQUIRED_SENSITIVITY)); threshold = float(thr[idx])
pred_val = (p_val >= threshold).astype(int)
sens = ((pred_val == 1) & (y_val == 1)).sum() / (y_val == 1).sum(); spec = ((pred_val == 0) & (y_val == 0)).sum() / (y_val == 0).sum()
print(f"threshold {threshold:.3f} | validation sensitivity {sens:.3f} | specificity {spec:.3f}")
assert sens >= REQUIRED_SENSITIVITY, "the requirement does not hold on validation"

## 5 · The test set, once

In [ ]:
FROZEN = {"model": "small CNN, seed %d, %d epochs, %.0f%% of the labels" % (SEED, EPOCHS, 100 * LABEL_FRACTION), "threshold": threshold}
_test_runs = []
def final_test_report():
    """THE ONLY CELL THAT TOUCHES THE TEST SET. Run once."""
    _test_runs.append(1)
    if len(_test_runs) > 1: print("!! the test set has been used", len(_test_runs), "times: it is no longer a test set")
    p = predict(model, X["test"]); y = Y["test"].numpy(); pr = (p >= FROZEN["threshold"]).astype(int)
    sens_f = lambda yy, ss: ((ss >= FROZEN["threshold"]) & (yy == 1)).sum() / max((yy == 1).sum(), 1)
    spec_f = lambda yy, ss: ((ss < FROZEN["threshold"]) & (yy == 0)).sum() / max((yy == 0).sum(), 1)
    res = {"AUC": roc_auc_score(y, p), "sensitivity": sens_f(y, p), "specificity": spec_f(y, p)}
    ci = {"AUC": bootstrap_ci(roc_auc_score, p, y), "sensitivity": bootstrap_ci(sens_f, p, y), "specificity": bootstrap_ci(spec_f, p, y)}
    return res, ci, (p, y)
test_res, test_ci, (p_test, y_test) = final_test_report()      # RUN ONCE
pd.DataFrame({k: {"value": round(float(test_res[k]), 3), "95% interval": f"[{test_ci[k][0]:.3f}, {test_ci[k][1]:.3f}]"} for k in test_res}).T

In [ ]:
assert len(_test_runs) == 1, "the test set was used more than once: restart the runtime and run all"
print("test set used once")

## 6 · Error analysis: the ten most confident wrong predictions (validation)

In [ ]:
wrong = np.where(pred_val != y_val)[0]; conf_wrong = wrong[np.argsort(-np.abs(p_val[wrong] - threshold))][:10]
fig, ax = plt.subplots(2, 5, figsize=(10, 4.4))
for a, i in zip(ax.ravel(), conf_wrong):
    a.imshow(X["val"][i, 0], cmap="gray"); a.set_title(f"true {y_val[i]} | P {p_val[i]:.2f}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()
print("errors by true class:", {int(c): int((y_val[wrong] == c).sum()) for c in (0, 1)}, "| false negatives are the dangerous ones if the requirement is sensitivity")

**TODO(team):** one sentence per observation (labelling problems? borderline cases? artefacts or markers? same kind of image each time?). Write them in this cell, then keep them in the model card.

## 7 · Explanation and sanity check (Grad-CAM; see NB09)
The function below is the one from NB09, written for any model and layer. Use it on three correct and three wrong validation cases, then **randomise the weights** and check that the maps change.

In [ ]:
from scipy.stats import spearmanr
def grad_cam(model, x, layer, sign=1.0):
    model.eval(); store = {}
    def fwd_hook(module, inp, out):
        store["a"] = out; out.register_hook(lambda g: store.__setitem__("g", g))
    h = layer.register_forward_hook(fwd_hook)
    try:
        logits = model(x.to(device))[:, 0]; model.zero_grad(); (sign * logits).sum().backward()
    finally: h.remove()
    a, g = store["a"].detach(), store["g"].detach()
    cam = F.relu((g.mean(dim=(2, 3), keepdim=True) * a).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    mx = cam.amax(dim=(1, 2), keepdim=True); return torch.where(mx > 0, cam / mx.clamp_min(1e-8), cam).cpu().numpy()

conf = np.abs(p_val - threshold); right = np.where(pred_val == y_val)[0]
pick = list(right[np.argsort(-conf[right])][:3]) + list(wrong[np.argsort(-conf[wrong])][:3])
def maps(m): return np.stack([grad_cam(m, X["val"][i:i + 1], m[LAST_CONV_RELU], 1.0 if pred_val[i] == 1 else -1.0)[0] for i in pick])
cam_t = maps(model)
rnd = copy.deepcopy(model)
for m_ in rnd.modules():
    if hasattr(m_, "reset_parameters"): m_.reset_parameters()
cam_r = maps(rnd)
corr = [spearmanr(cam_t[k].ravel(), cam_r[k].ravel())[0] for k in range(6)]
fig, ax = plt.subplots(3, 6, figsize=(12, 6.2))
for k, i in enumerate(pick):
    for r, (im, ttl) in enumerate([(None, "image"), (cam_t[k], "trained"), (cam_r[k], "randomised")]):
        ax[r, k].imshow(X["val"][i, 0], cmap="gray")
        if im is not None: ax[r, k].imshow(im, cmap="jet", alpha=0.45)
        ax[r, k].axis("off"); ax[r, k].set_title(f"{ttl} | true {y_val[i]}" if r == 0 else ttl, fontsize=8)
plt.tight_layout(); plt.show()
print("Spearman correlation trained vs randomised (mean):", round(float(np.nanmean(corr)), 2))
assert np.nanmean(corr) < 0.9, "the maps do not depend on the weights: this method does not explain your model"

**TODO(team):** 3 sentences: where is the heat? Did the map change after randomisation? What can this explanation **not** tell you? (Not causal; plausible is not proof; shortcuts look fine.)

## 8 · Shift or subgroup check, with intervals

In [ ]:
def apply_shift(x, kind, seed=0):
    g = torch.Generator().manual_seed(seed)
    if kind == "clean": return x.clone()
    if kind == "brighter": return (x + 0.15).clamp(0, 1)
    if kind == "darker": return (x - 0.15).clamp(0, 1)
    if kind == "noise": return (x + 0.08 * torch.randn(x.shape, generator=g)).clamp(0, 1)
    if kind == "blur": return F.avg_pool2d(x, 5, 1, 2, count_include_pad=False)
    if kind == "lowres":
        h = x.shape[-1]; return F.interpolate(F.interpolate(x, size=(max(h // 4, 2),) * 2, mode="bilinear", align_corners=False), size=(h, h), mode="bilinear", align_corners=False).clamp(0, 1)
    raise ValueError(kind)
rows = []
for kind in ["clean", "brighter", "darker", "noise", "blur", "lowres"]:
    p = predict(model, apply_shift(X["val"], kind)); lo_, hi_ = bootstrap_ci(roc_auc_score, p, y_val)
    rows.append(dict(shift=kind, AUC=round(roc_auc_score(y_val, p), 3), ci95=f"[{lo_:.3f}, {hi_:.3f}]", sensitivity_at_frozen_threshold=round(float(((p >= threshold) & (y_val == 1)).sum() / (y_val == 1).sum()), 3)))
shift_table = pd.DataFrame(rows); shift_table.to_csv("outputs/shift_check.csv", index=False); shift_table

**TODO(team):** which shift hurts most, and is the change bigger than the interval? If your dataset has metadata (sex, age band, scanner), do the same by **subgroup** instead. Report the result even if it is small.

### Optional · calibration (question C) and your own experiment
`ece()` is the expected calibration error from NB06. Use it, add temperature scaling, or add the experiment your question needs (label fractions, a second model, a second dataset). **Keep one trivial baseline and one interval next to every number.**

In [ ]:
def ece(probs, labels, n_bins=10):
    bins = np.minimum((probs * n_bins).astype(int), n_bins - 1); tot = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any(): tot += m.mean() * abs(labels[m].mean() - probs[m].mean())
    return float(tot)
print("ECE on validation:", round(ece(p_val, y_val), 3))

# TODO(team): YOUR EXPERIMENT. Examples: for frac in [1.0, 0.1, 0.01]: train + validation AUC with interval;
# fine-tuned ResNet (see NB07) vs this CNN; temperature scaling (see NB06). Always report an interval and the trivial baseline.
experiment_notes = "TODO(team) what you did and what it showed, in 3 sentences"

### Question B (Montgomery lung segmentation)
Start from **NB08** (U-Net, loss, Dice) and keep this notebook's *structure*: question, split, metric with interval, baseline, error analysis, model card. The helper below reports Dice **by group** (normal vs TB) with a bootstrap interval. Montgomery has 138 images and **one image per patient is not verified**, so a patient-level split cannot be guaranteed: state it.

In [ ]:
def dice_by_group(dice, groups, n_boot=1000, seed=0):
    """dice: per-image Dice (n,), groups: array of group labels (n,). Returns a table with mean and 95% bootstrap interval per group."""
    rng = np.random.default_rng(seed); rows = []
    for g_ in np.unique(groups):
        d = np.asarray(dice)[np.asarray(groups) == g_]
        means = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
        rows.append(dict(group=g_, n=len(d), mean_dice=round(float(d.mean()), 3), ci95=f"[{np.percentile(means, 2.5):.3f}, {np.percentile(means, 97.5):.3f}]"))
    return pd.DataFrame(rows)
# usage (after NB08):  dice_by_group(dt, D["tb"][te])      # tb: 0 normal, 1 TB
print(dice_by_group(np.r_[np.full(30, .95), np.full(30, .90) + np.random.default_rng(0).normal(0, .02, 30)], np.r_[np.zeros(30), np.ones(30)]))

## 9 · Model card
Fill every field. Rules: **intended use** = task, population, user; **not intended for** = where it must not be used; the **metrics have intervals and the trivial baseline**; **limits** come from your error analysis and your checks, not from imagination. Mitchell et al. (2019), *Model Cards for Model Reporting*.

In [ ]:
CARD = {
    "Model": "TODO(team)",
    "Version and date": "TODO(team)",
    "Intended use": "TODO(team) task, population, user",
    "Not intended for": "TODO(team) where it must NOT be used",
    "Data and split": "TODO(team) source, size, how it was split, duplicates checked?",
    "Metrics (test, once)": "TODO(team) main metric WITH interval, next to the trivial baseline",
    "Subgroup or shift result": "TODO(team) what you checked and found",
    "Failure modes and limits": "TODO(team) from your error analysis and your checks",
    "Ethical considerations": "TODO(team) who could be harmed? who is missing from the data?",
}

In [ ]:
empty = [k for k, v in CARD.items() if "TODO" in str(v) or not str(v).strip()]
assert not empty, f"fill these model-card fields: {empty}"
card_md = "# Model card\n\n" + "\n\n".join(f"**{k}.** {v}" for k, v in CARD.items()) + "\n"
Path("outputs/model_card.md").write_text(card_md, encoding="utf-8"); print(card_md)

## Reproducibility self-check (the peer will run the same list tomorrow)
Run it last. The *Restart and run all* test is yours: do it now, on a clean runtime, and fix whatever breaks.

In [ ]:
checks = {
    "seed is fixed": "SEED" in globals(),
    "no absolute paths in the saved files": "/content" not in Path("outputs/split.json").read_text() and "/home" not in Path("outputs/split.json").read_text(),
    "split saved": Path("outputs/split.json").exists(),
    "split assertion ran (no duplicates between sets)": sum(dups.values()) == 0,
    "test set used once": len(_test_runs) == 1,
    "metric has an interval and a trivial baseline": "AUC" in test_ci and maj_acc_val > 0,
    "explanation and sanity check ran": float(np.nanmean(corr)) < 0.9,
    "shift or subgroup table saved": Path("outputs/shift_check.csv").exists(),
    "model card written": Path("outputs/model_card.md").exists(),
    "team and question filled": "TODO" not in (str(TEAM) + QUESTION),
}
for k, v in checks.items(): print("[x]" if v else "[ ]", k)
assert all(checks.values()), "some checks fail: fix them before you submit"
print("\nSelf-check passed (%d of %d). Now: Runtime > Restart session and run all." % (sum(checks.values()), len(checks)))

## Individual contribution statement
One line per member (**what each person did**: code, data, evaluation, interpretation, writing). Submitted with the notebook.

| Member | Contribution |
|---|---|
| TODO(team) | TODO(team) |

## Submission
One notebook per team, **Sunday 22 November 2026, 23:59 (Madrid)**, on Moodle. Name it `NB10_team<number>.ipynb`. Do **not** include the Montgomery images or any restricted data; the notebook must download or ask for what it needs and say how.